# Byte-pair encoding

**Track:** Training LLMs. **Level:** Intermediate. **CPU:** yes.

## Goals

- Train a byte-level BPE merge table on a small corpus.
- Encode and decode back to the same Unicode string, including a Hebrew sentence.
- Compare the token count with the byte count on repeated text.

## Sources and license

Original implementation of the algorithm in Sennrich, Haddow, and Birch, *Neural Machine Translation of Rare Words with Subword Units* (2016). The training loop follows the design of Karpathy's minbpe (MIT). That repository is not copied. Byte-level BPE is the scheme used by GPT-2.

## Concepts

The corpus is a sequence of bytes, so any Unicode string is in vocabulary from the start. Each merge replaces the most frequent adjacent pair with a new id. Encoding applies the learned merges from earliest to latest, which is the lowest id among the pairs that still occur. Decoding walks the merge table backwards into bytes. Compression comes from merges that repeat. A language with multi-byte characters starts further from one token per character, which is a property of UTF-8, not of the merge rule.


In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", 'numpy'])
print("packages ready")


In [ ]:
import os
import sys
from pathlib import Path

def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for cand in [start, *start.parents]:
        if (cand / "labs" / "_shared" / "llm.py").exists():
            return cand
    if "google.colab" in sys.modules:
        root = Path("/content/ShaiLearnsAI")
        marker = root / "labs" / "_shared" / "llm.py"
        if not marker.exists():
            os.system(
                "git clone --depth 1 https://github.com/shai200/ShaiLearnsAI.git /content/ShaiLearnsAI"
            )
        return root
    raise RuntimeError(
        "Clone https://github.com/shai200/ShaiLearnsAI and run this notebook from that directory."
    )

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")
print("repo", ROOT)

from collections import Counter

from labs._shared.seed import seed_everything

seed_everything(0)


## Guided exercise: train, encode, decode


In [ ]:
def pair_counts(seqs):
    counts = Counter()
    for seq in seqs:
        for left, right in zip(seq, seq[1:]):
            counts[(left, right)] += 1
    return counts


def apply_merge(seq, pair, new_id):
    out = []
    i = 0
    while i < len(seq):
        if i < len(seq) - 1 and (seq[i], seq[i + 1]) == pair:
            out.append(new_id)
            i += 2
        else:
            out.append(seq[i])
            i += 1
    return out


def train_bpe(text: str, num_merges: int) -> dict:
    seqs = [list(text.encode("utf-8"))]
    merges = {}
    for i in range(num_merges):
        stats = pair_counts(seqs)
        if not stats:
            break
        pair = max(stats, key=stats.get)
        new_id = 256 + i
        seqs = [apply_merge(seq, pair, new_id) for seq in seqs]
        merges[pair] = new_id
    return merges


def encode(text: str, merges: dict) -> list[int]:
    seq = list(text.encode("utf-8"))
    while True:
        stats = pair_counts([seq])
        present = [pair for pair in stats if pair in merges]
        if not present:
            break
        pair = min(present, key=lambda item: merges[item])
        seq = apply_merge(seq, pair, merges[pair])
    return seq


def decode(ids: list[int], merges: dict) -> str:
    vocab = {i: bytes([i]) for i in range(256)}
    for (left, right), new_id in sorted(merges.items(), key=lambda item: item[1]):
        vocab[new_id] = vocab[left] + vocab[right]
    return b"".join(vocab[i] for i in ids).decode("utf-8")


corpus = ("the cat sat on the mat. the cat sat on the hat. " * 40)
merges = train_bpe(corpus, 30)
roundtrip = decode(encode(corpus, merges), merges)
print("merges", len(merges), "bytes", len(corpus.encode()), "tokens", len(encode(corpus, merges)))
print("first merges", list(merges.items())[:5])
assert roundtrip == corpus
assert len(encode(corpus, merges)) < len(corpus.encode())

hebrew = "חשבון החיסכון מניב ריבית שנתית"
hebrew_merges = train_bpe(hebrew * 6, 15)
assert decode(encode(hebrew, hebrew_merges), hebrew_merges) == hebrew
assert len(hebrew.encode("utf-8")) > len(hebrew)
print("LAB OK")


## Your turn

Encode the English sentence and the Hebrew sentence with a merge table trained on the concatenated parallel text. Both must round-trip. Print tokens per whitespace-separated word. That ratio is called fertility in the tokenization literature.

## Solution


In [ ]:
parallel = "the savings account yields interest. חשבון החיסכון מניב ריבית. " * 10
merges_p = train_bpe(parallel, 40)
en = "the savings account yields interest."
he = "חשבון החיסכון מניב ריבית."
for label, sentence in (("en", en), ("he", he)):
    ids = encode(sentence, merges_p)
    words = sentence.split()
    print(label, "tokens", len(ids), "words", len(words), "fertility", round(len(ids) / len(words), 2))
    assert decode(ids, merges_p) == sentence


## Beyond the course

Train two merge tables, one on English only and one on the parallel text, and encode the Hebrew sentence with the English-only table. It still round-trips, because unknown bytes stay bytes. The token count is the thing that changes. A tokenizer that never saw Hebrew spends an id per byte.


In [ ]:
english_only = train_bpe("the savings account yields interest. " * 30, 40)
ids_en_table = encode(he, english_only)
ids_bi_table = encode(he, merges_p)
print("hebrew tokens, english merges", len(ids_en_table), "parallel merges", len(ids_bi_table))
assert decode(ids_en_table, english_only) == he
assert len(ids_en_table) >= len(ids_bi_table)


## Research questions

1. Fertility was higher for one of the two sentences. Is that the language, the script, or the fact that the parallel corpus contained more English repeats?
2. GPT-2's byte-level BPE has a regex pre-tokenizer that keeps most punctuation separate. What bug appears in this lab's encoder because it does not?
3. A merge learned from one domain will happily segment another domain into bytes. Where does that show up in a multilingual model's loss, as opposed to in the round-trip test?

## Further reading

- Sennrich, Haddow, and Birch, *Neural Machine Translation of Rare Words with Subword Units* (ACL 2016).
- Karpathy, minbpe (MIT). https://github.com/karpathy/minbpe. Design reference, not copied.
- Radford et al., *Language Models are Unsupervised Multitask Learners* (GPT-2, 2019), on byte-level BPE.
